# In Class Activity April 14th, 2026

In [12]:
# pip install optuna

### Importing libraries, preparing data, initial EDA

In [13]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [14]:
# importing data
adult = pd.read_csv('../datasets/adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [15]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


Done! Use 'show' commands to display/save.   |██████████| [100%]   00:00 -> (00:00 left)


Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





- The EDA revealed several important patterns. The dataset is class-imbalanced, as roughly 75% of records are <= 50K while 25% are > 50K, which means we should use `scale_pos_weight` in XGBoost to compensate for this. 

- `educational-num`, `marital-status`, `relationship`, `capital-gain`, and `hours-per-week` show the strongest associations with income. 

- `capital-gain` is highly skewed with most values at zero, but large values are very predictive of > 50K. 

- About 7% of records have missing values in `workclass`, `occupation`, and `native-country` (encoded as NaN after replacing `?`); XGBoost is built to handle these.

- `education` and `educational-num` are redundant, so I dropped `education`. 

- Finally, because of the class imbalance, F1 score is the better evaluation metric rather than accuracy.

### Data Preprocessing (minimal) and Baseline Model

In [16]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

C:\Users\ivpri\AppData\Local\Temp\ipykernel_39140\2764229057.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = adult.select_dtypes(include='object').columns


,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [17]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=42, stratify=y)

In [18]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.70680507 0.70892566 0.70898981 0.72424942 0.71086556]
Mean F1 score: 0.7119671046056588


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

- The default XGBoost model achieved a mean cross-validated F1 score of 0.7120; the fold scores were 0.7068, 0.7089, 0.7090, 0.7242, 0.7109. 
- This is a decent baseline, but we have room to improve. 
- Adding `scale_pos_weight=3.18` bumped the mean F1 to 0.7146. 
- `max_depth=5` performed best (0.7148) among depths tested, and `learning_rate=0.2` was the strongest learning rate (0.7157). 
- `subsample` had minimal impact across 0.6–1.0. 
- The combined best exploration model (`scale_pos_weight=3.18`, `max_depth=5`, `learning_rate=0.1`, `subsample=0.8`) scored 0.7113, which is slightly lower than the individual best settings, suggesting that combination wasn't optimal. 
- We can try systematic tuning with GridSearchCV, RandomizedSearchCV, and Optuna to find a better joint configuration.

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [19]:
# Exploring XGBoost features
# Feature 1: scale_pos_weight — addresses class imbalance
# The dataset has ~3x more <=50K than >50K, so scale_pos_weight ~3 is a good starting point
neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos
print(f'Suggested scale_pos_weight: {spw:.2f}')

xgb_spw = XGBClassifier(enable_categorical=True, random_state=42, scale_pos_weight=spw)
scores_spw = cross_val_score(xgb_spw, X, y, cv=skf, scoring='f1')
print(f'scale_pos_weight={spw:.2f} | Mean F1: {scores_spw.mean():.4f}')

# Feature 2: max_depth — controls tree complexity
for depth in [3, 5, 7, 9]:
    xgb_d = XGBClassifier(enable_categorical=True, random_state=42,
                          scale_pos_weight=spw, max_depth=depth)
    s = cross_val_score(xgb_d, X, y, cv=skf, scoring='f1')
    print(f'max_depth={depth} | Mean F1: {s.mean():.4f}')

# Feature 3: learning_rate — shrinks contribution of each tree
for lr in [0.01, 0.05, 0.1, 0.2]:
    xgb_lr = XGBClassifier(enable_categorical=True, random_state=42,
                           scale_pos_weight=spw, max_depth=5, learning_rate=lr)
    s = cross_val_score(xgb_lr, X, y, cv=skf, scoring='f1')
    print(f'learning_rate={lr} | Mean F1: {s.mean():.4f}')

# Feature 4: subsample — fraction of training samples used per tree (reduces overfitting)
for ss in [0.6, 0.8, 1.0]:
    xgb_ss = XGBClassifier(enable_categorical=True, random_state=42,
                           scale_pos_weight=spw, max_depth=5, learning_rate=0.1, subsample=ss)
    s = cross_val_score(xgb_ss, X, y, cv=skf, scoring='f1')
    print(f'subsample={ss} | Mean F1: {s.mean():.4f}')

# Best model from exploration: scale_pos_weight + max_depth=5 + learning_rate=0.1 + subsample=0.8
xgb_best_explore = XGBClassifier(enable_categorical=True, random_state=42,
                                  scale_pos_weight=spw, max_depth=5,
                                  learning_rate=0.1, subsample=0.8)
scores_best = cross_val_score(xgb_best_explore, X, y, cv=skf, scoring='f1')
print(f'\nBest exploration model | Mean F1: {scores_best.mean():.4f}')


Suggested scale_pos_weight: 3.18
scale_pos_weight=3.18 | Mean F1: 0.7146
max_depth=3 | Mean F1: 0.7123
max_depth=5 | Mean F1: 0.7148
max_depth=7 | Mean F1: 0.7140
max_depth=9 | Mean F1: 0.7129
learning_rate=0.01 | Mean F1: 0.6697
learning_rate=0.05 | Mean F1: 0.6988
learning_rate=0.1 | Mean F1: 0.7126
learning_rate=0.2 | Mean F1: 0.7157
subsample=0.6 | Mean F1: 0.7116
subsample=0.8 | Mean F1: 0.7113
subsample=1.0 | Mean F1: 0.7126

Best exploration model | Mean F1: 0.7113


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [20]:
# GridSearchCV — tuning 5 hyperparameters
param_grid = {
    'scale_pos_weight': [2.0, 3.0, 4.0],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.9],
    'n_estimators': [100, 200]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, enable_categorical=True),
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
print(f'Best parameters from GridSearchCV: {xgb_grid.best_params_}')
print(f'Best CV F1 score from GridSearchCV: {xgb_grid.best_score_:.4f}')

# Final model with best params
xgb_grid_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    **xgb_grid.best_params_
)
xgb_grid_best.fit(X_train, y_train)
y_pred_grid = xgb_grid_best.predict(X_test)
print(f'Classification report for GridSearchCV-tuned model:\n{classification_report(y_test, y_pred_grid)}')


Best parameters from GridSearchCV: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100, 'scale_pos_weight': 2.0, 'subsample': 0.9}
Best CV F1 score from GridSearchCV: 0.7279
Classification report for GridSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.79      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [21]:
# tuning xgboost classifier with RandomizedSearchCV
param_dist = {
    'scale_pos_weight': np.linspace(1.0, 10.0, num=10),
    'max_depth': np.arange(3, 11),
    'learning_rate': np.linspace(0.01, 0.3, num=10),
    'subsample': np.linspace(0.5, 1.0, num=6),
    'n_estimators': np.arange(100, 401, 100)
}

# using preferred model config from feature exploration (scale_pos_weight, max_depth, learning_rate, subsample)
xgb_random = RandomizedSearchCV(XGBClassifier(random_state=42, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=42)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build final model using best parameters from RandomizedSearchCV and evaluate on the test set
xgb_random_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    scale_pos_weight=xgb_random.best_params_['scale_pos_weight'],
    max_depth=xgb_random.best_params_['max_depth'],
    learning_rate=xgb_random.best_params_['learning_rate'],
    subsample=xgb_random.best_params_['subsample'],
    n_estimators=xgb_random.best_params_['n_estimators']
)
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'subsample': np.float64(1.0), 'scale_pos_weight': np.float64(2.0), 'n_estimators': np.int64(100), 'max_depth': np.int64(8), 'learning_rate': np.float64(0.07444444444444444)}
Best F1 score from RandomizedSearchCV: 0.7257037867594173
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      7431
           1       0.67      0.80      0.73      2338

    accuracy                           0.86      9769
   macro avg       0.80      0.84      0.82      9769
weighted avg       0.87      0.86      0.86      9769



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [22]:
# tuning with Optuna — same 5 hyperparameters as GridSearchCV and RandomizedSearchCV
def objective(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 10.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    n_estimators = trial.suggest_int('n_estimators', 100, 400, step=100)
    
    xgb_optuna = XGBClassifier(
        random_state=42,
        enable_categorical=True,
        scale_pos_weight=scale_pos_weight,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        n_estimators=n_estimators
    )
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build final model with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=42, scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  enable_categorical=True)
xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 15:13:34,835] A new study created in memory with name: no-name-7a501ada-d30f-4cad-97ce-45ac7d8c092b
Best trial: 0. Best value: 0.71487:   5%|▌         | 1/20 [00:16<05:14, 16.53s/it]

[I 2026-04-15 15:13:51,368] Trial 0 finished with value: 0.7148697537959791 and parameters: {'scale_pos_weight': 2.802901324301373, 'max_depth': 9, 'learning_rate': 0.016391578037933993, 'subsample': 0.5818295294686993, 'n_estimators': 200}. Best is trial 0 with value: 0.7148697537959791.


Best trial: 0. Best value: 0.71487:  10%|█         | 2/20 [00:32<04:51, 16.17s/it]

[I 2026-04-15 15:14:07,290] Trial 1 finished with value: 0.689361092414621 and parameters: {'scale_pos_weight': 9.317764906082177, 'max_depth': 7, 'learning_rate': 0.2322005932651512, 'subsample': 0.5247961615335132, 'n_estimators': 300}. Best is trial 0 with value: 0.7148697537959791.


Best trial: 0. Best value: 0.71487:  15%|█▌        | 3/20 [00:37<03:10, 11.19s/it]

...

[I 2026-04-15 15:17:30,825] Trial 17 finished with value: 0.7170843020916683 and parameters: {'scale_pos_weight': 1.902431955172632, 'max_depth': 8, 'learning_rate': 0.06331062706563106, 'subsample': 0.5584872549370834, 'n_estimators': 400}. Best is trial 11 with value: 0.7253018324975281.


Best trial: 11. Best value: 0.725302:  95%|█████████▌| 19/20 [04:07<00:12, 12.81s/it]

[I 2026-04-15 15:17:42,831] Trial 18 finished with value: 0.7038836404062454 and parameters: {'scale_pos_weight': 3.990089327793355, 'max_depth': 10, 'learning_rate': 0.17240075584686798, 'subsample': 0.6535788195234374, 'n_estimators': 200}. Best is trial 11 with value: 0.7253018324975281.


Best trial: 11. Best value: 0.725302: 100%|██████████| 20/20 [04:11<00:00, 12.55s/it]


[I 2026-04-15 15:17:45,846] Trial 19 finished with value: 0.6875981013529799 and parameters: {'scale_pos_weight': 3.887099973111919, 'max_depth': 5, 'learning_rate': 0.04771973374591463, 'subsample': 0.7563375583037112, 'n_estimators': 100}. Best is trial 11 with value: 0.7253018324975281.
Best parameters from Optuna: {'scale_pos_weight': 1.4347917549682916, 'max_depth': 9, 'learning_rate': 0.01851679808976204, 'subsample': 0.630088513002124, 'n_estimators': 400}
Best F1 score from Optuna: 0.7253018324975281
Classification report for Optuna-tuned model:
              precision    recall  f1-score   support

           0       0.90      0.93      0.92      7431
           1       0.76      0.67      0.71      2338

    accuracy                           0.87      9769
   macro avg       0.83      0.80      0.81      9769
weighted avg       0.87      0.87      0.87      9769



### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


For all three methods, I tuned the same 5 hyperparameters (`scale_pos_weight`, `max_depth`, `learning_rate`, `subsample`, `n_estimators`) with 5-fold stratified cross-validation.

- GridSearchCV produced the highest CV F1 of 0.7279, with best params `learning_rate=0.1`, `max_depth=6`, `n_estimators=100`, `scale_pos_weight=2.0`, `subsample=0.9`. The classification report showed F1 of 0.73 for the >50K class (precision 0.67, recall 0.79), with 86% overall accuracy. It was the most exhaustive approach since it searched all 108 combinations, but also the slowest at almost 10 minutes to run.

- RandomizedSearchCV achieved a CV F1 of 0.7257, with best params `subsample=1.0`, `scale_pos_weight=2.0`, `n_estimators=100`, `max_depth=8`, `learning_rate=0.074`. The report also showed an f1 score of 0.73 for the >50K class (precision 0.67, recall 0.80), 86% overall accuracy, nearly identical to GridSearchCV but found in 20 random iterations instead of 108.

- Optuna achieved a CV F1 of 0.7253, with best params `scale_pos_weight=1.43`, `max_depth=9`, `learning_rate=0.019`, `subsample=0.630`, `n_estimators=400`. The report showed F1 of 0.71 for the >50K class (precision 0.76, recall 0.67), 87% overall accuracy. It seems like Optuna's model traded recall for precision compared to the other two; it was more conservative about predicting >50K.

- GridSearchCV produced the best CV F1 overall (0.7279), even though all three methods were very close. In terms of preference, RandomizedSearchCV offers the best of all worlds: it matched GridSearchCV's test performance with far less compute time. Optuna is probably more fit for larger search spaces and more trials, but with only 20 trials it didn't outperform the others here.